# CoCoBun Backend — API Fixes E2E (Sections 1–4)

Live API walkthrough for the frontend team. Each cell prints the **exact request**
(`→ METHOD url`, params, body) and the **real response** (`← status` + JSON), plus a
✅/❌ check. Write operations **save and restore** original state, so it's safe and
repeatable against a shared environment.

逐条展示请求/响应;写操作会先存后还原,可安全重复运行。

**Run:** set env `FIREBASE_ID_TOKEN` (and optionally `FEEDS_BASE_URL`), then Run All.

## 0. Config

In [1]:
import os
FEEDS_BASE_URL = os.environ.get("FEEDS_BASE_URL", "http://34.44.73.189:8003")  # PROD feeds
# Provide a Firebase ID token via env (recommended) or paste it here:
FIREBASE_ID_TOKEN = os.environ.get("FIREBASE_ID_TOKEN", "")

## 1. Helpers — `show()` prints request + response

In [2]:
from __future__ import annotations
import json, textwrap, urllib.error, urllib.parse, urllib.request

BASE = FEEDS_BASE_URL
AUTH = {"Authorization": f"Bearer {FIREBASE_ID_TOKEN}"}
RESULTS = []

def record(name, ok, detail=""):
    RESULTS.append((name, ok)); print(("  \u2705 " if ok else "  \u274c ") + name + (f"  ({detail})" if detail else ""))
def skip(name, detail=""):
    RESULTS.append((name, None)); print("  \u23ed\ufe0f  " + name + (f"  ({detail})" if detail else ""))

def _http(method, url, body=None):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method=method,
                                 headers={"Content-Type": "application/json", **AUTH})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, json.loads(r.read().decode() or "{}")
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode(errors="replace") or "{}")

def _compact(v, list_max=2, str_max=96):
    if isinstance(v, dict):  return {k: _compact(x, list_max, str_max) for k, x in v.items()}
    if isinstance(v, list):
        out = [_compact(x, list_max, str_max) for x in v[:list_max]]
        if len(v) > list_max: out.append(f"...(+{len(v)-list_max} more)")
        return out
    if isinstance(v, str) and len(v) > str_max: return v[:str_max] + "\u2026"
    return v

def show(method, path, body=None):
    """Print the exact request (URL/params/body) and response; return (status, full_json)."""
    url = BASE + path
    print(f"\u2192 {method} {url}")
    if body is not None: print(f"  body: {json.dumps(body, ensure_ascii=False)}")
    st, data = _http(method, url, body=body)
    print(f"\u2190 {st}")
    print(textwrap.indent(json.dumps(_compact(data), indent=2, ensure_ascii=False), "  "))
    print()
    return st, data

def qparams(url):  # signing query params
    return dict(urllib.parse.parse_qsl(urllib.parse.urlsplit(url).query))

print("helpers ready; BASE =", BASE)

helpers ready; BASE = http://34.44.73.189:8003


## 2. Auth + health

In [3]:
assert FIREBASE_ID_TOKEN, "Set env FIREBASE_ID_TOKEN (or paste it in the Config cell)"
st, _ = show("GET", "/health")
assert st == 200, "feeds unreachable"

→ GET http://34.44.73.189:8003/health


← 200
  {
    "status": "ok"
  }



## Section 1 — Library

In [4]:
st, b = show("GET", "/api/v1/feeds/stories/grouped?limit=3")
record("1.1 GET /stories/grouped returns 200 (was HTTP 500)", st==200 and isinstance(b.get("groups"), list), f"{len(b.get('groups',[]))} categories")

→ GET http://34.44.73.189:8003/api/v1/feeds/stories/grouped?limit=3


← 200
  {
    "groups": [
      {
        "category_key": "originals",
        "category_label": "CoCo's Originals",
        "items": [
          {
            "id": "92116490-7ae3-49ec-9c95-456b445ee66d",
            "title": "E2E Test Original 1779084309",
            "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/eb323335-bb30-4782-b9fa-d…",
            "summary": "Acceptance run — pages mirrored from a real prod story.",
            "category": "literature",
            "is_original": true,
            "uploader": {
              "user_id": "cocobun_official",
              "nickname": "CoCoBun",
              "avatar_url": null
            },
            "like_count": 2,
            "view_count": 7,
            "is_liked": true,
            "created_at": "2026-05-18T06:05:09.626948Z"
          },
          {
            "id": "41b55972-b3a2-47bd-90a1-0740fd3d70a4",
            "title": "E2E Test Original 1779084131",
            "cover_i

In [5]:
# 1.2 verify search actually filters by TITLE (derive a real term from live data)
_, sample = show("GET", "/api/v1/feeds/stories?limit=5")
term = next((w.lower() for it in sample.get("items", []) for w in it["title"].split() if len(w) >= 4), "")
if term:
    st, b = show("GET", f"/api/v1/feeds/stories?search={term}&limit=20")
    titles = [i["title"].lower() for i in b.get("items", [])]
    record(f"1.2 search='{term}' filters by title (all results contain it)",
           st==200 and len(titles) >= 1 and all(term in t for t in titles), f"{len(titles)} items match")
else:
    skip("1.2 search", "no story data to derive a term")

→ GET http://34.44.73.189:8003/api/v1/feeds/stories?limit=5


← 200
  {
    "items": [
      {
        "id": "423d6cc3-7f42-4a9c-a1ca-caf65a45dca1",
        "title": "Princess Lily and the Royal Gardens",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/eb323335-bb30-4782-b9fa-d…",
        "summary": "A graceful young deer explores the palace gardens and meets a friend in need.",
        "category": "humor_interactive",
        "is_original": true,
        "uploader": {
          "user_id": "cocobun_official",
          "nickname": "CoCoBun",
          "avatar_url": null
        },
        "like_count": 1,
        "view_count": 11,
        "is_liked": false,
        "created_at": "2026-05-18T05:51:50.241436Z"
      },
      {
        "id": "92116490-7ae3-49ec-9c95-456b445ee66d",
        "title": "E2E Test Original 1779084309",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/eb323335-bb30-4782-b9fa-d…",
        "summary": "Acceptance run — pages mi

← 200
  {
    "items": [
      {
        "id": "423d6cc3-7f42-4a9c-a1ca-caf65a45dca1",
        "title": "Princess Lily and the Royal Gardens",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/eb323335-bb30-4782-b9fa-d…",
        "summary": "A graceful young deer explores the palace gardens and meets a friend in need.",
        "category": "humor_interactive",
        "is_original": true,
        "uploader": {
          "user_id": "cocobun_official",
          "nickname": "CoCoBun",
          "avatar_url": null
        },
        "like_count": 1,
        "view_count": 11,
        "is_liked": false,
        "created_at": "2026-05-18T05:51:50.241436Z"
      }
    ],
    "next_cursor": null,
    "has_more": false
  }

  ✅ 1.2 search='princess' filters by title (all results contain it)  (1 items match)


In [6]:
st, b = show("GET", "/api/v1/feeds/my-uploads/grouped")
record("1.3 GET /my-uploads/grouped (grouped by category)", st==200 and isinstance(b.get("groups"), list))

→ GET http://34.44.73.189:8003/api/v1/feeds/my-uploads/grouped


← 200
  {
    "groups": [
      {
        "category_key": "adventure_fantasy",
        "category_label": "Adventure & Fantasy",
        "items": [
          {
            "id": "36475aa6-7c7f-4bb5-8584-ed946b5b55ce",
            "title": "Notebook publish demo 1779089554",
            "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
            "category": "adventure_fantasy",
            "like_count": 0,
            "view_count": 0,
            "download_count": 1,
            "is_published": true,
            "created_at": "2026-05-18T07:32:34.888049Z"
          },
          {
            "id": "6aaff53e-c06a-4d71-8cc3-3ebac8e47ebd",
            "title": "A Little Lion Cub Bedtime Story",
            "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/e5e12956-654b-4446-ba91-2…",
            "category": "adventure_fantasy",
            "like_count": 1,
            "view_count

## Section 2 — Playlist

In [7]:
st, b = show("GET", "/api/v1/feeds/playlists?limit=2&offset=0")
pls = b.get("playlists", []); PID = next((p["id"] for p in pls if p.get("is_default")), pls[0]["id"]) if pls else None
record("2.1a pagination fields (total/limit/offset/has_more)", st==200 and all(k in b for k in ("total","limit","offset","has_more")))
record("2.1b cover_image_url on each card", bool(pls) and "cover_image_url" in pls[0])

→ GET http://34.44.73.189:8003/api/v1/feeds/playlists?limit=2&offset=0


← 200
  {
    "playlists": [
      {
        "id": "ad8e55d1-99e3-4cbf-97aa-f26c89f89605",
        "name": "My Playlist",
        "is_default": true,
        "item_count": 0,
        "cover_image_url": null,
        "created_at": "2026-05-17T05:16:17.352811Z"
      }
    ],
    "total": 1,
    "limit": 2,
    "offset": 0,
    "has_more": false
  }

  ✅ 2.1a pagination fields (total/limit/offset/has_more)
  ✅ 2.1b cover_image_url on each card


### Shared: pick one bookshelf entry (used by 2.2/2.3 + Section 3)

In [8]:
st, b = show("GET", "/api/v1/feeds/bookshelf?limit=3")
BS = b.get("items", []); ENTRY = BS[0] if BS else None; EID = ENTRY["id"] if ENTRY else None
record("3.2 list exposes is_favorite / is_liked / is_downloaded", bool(ENTRY) and all(k in ENTRY for k in ("is_favorite","is_liked","is_downloaded")))

→ GET http://34.44.73.189:8003/api/v1/feeds/bookshelf?limit=3


← 200
  {
    "items": [
      {
        "id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e",
        "title": "Notebook publish demo 1779089554",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
        "summary": "Live demo of the user-publish path: completed task → public library.",
        "source_type": "created",
        "is_downloaded": false,
        "is_favorite": false,
        "is_liked": false,
        "created_at": "2026-05-18T07:32:34.895933Z"
      },
      {
        "id": "518323cb-0145-449d-b639-4ac0f7aa9fc6",
        "title": "A Little Lion Cub Bedtime Story",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/e5e12956-654b-4446-ba91-2…",
        "summary": "My first published story — Little Prince Leo discovers his royal heritage.",
        "source_type": "created",
        "is_downloaded": false,
        "is_favorite": false,
        "is_liked": fa

In [9]:
# 2.3 add bookshelf -> playlist, then 2.2 verify detail has the item (state-safe: restore original)
if ENTRY and PID:
    _, det = _http("GET", BASE + f"/api/v1/feeds/bookshelf/{EID}")
    ids = [i for i in (EID, det.get("published_story_id")) if i]
    _, cur = _http("GET", BASE + f"/api/v1/feeds/playlists/{PID}?limit=100")
    was_present = any(it.get("story_id") in ids for it in cur.get("items", []))   # remember original state
    for r in ids: _http("DELETE", BASE + f"/api/v1/feeds/playlists/{PID}/items/{r}")
    st, added = show("POST", f"/api/v1/feeds/playlists/{PID}/items", {"bookshelf_id": EID})
    record("2.3 add bookshelf story to a playlist (bookshelf_id)", st==200 and isinstance(added.get("items"), list))
    st, detl = show("GET", f"/api/v1/feeds/playlists/{PID}?limit=2&offset=0")
    items = detl.get("items", [])
    record("2.2 item pagination + items[].source_type populated",
           st==200 and all(k in detl for k in ("total_items","has_more")) and len(items) >= 1
           and all(i.get("source_type") in ("library","bookshelf") for i in items), f"{len(items)} items")
    if not was_present:                                                            # restore: remove only if we added it
        for r in ids: _http("DELETE", BASE + f"/api/v1/feeds/playlists/{PID}/items/{r}")
    print(f"  (restored playlist: item present originally = {was_present})")
else:
    skip("2.2 / 2.3", "need a bookshelf entry + a playlist")

→ POST http://34.44.73.189:8003/api/v1/feeds/playlists/ad8e55d1-99e3-4cbf-97aa-f26c89f89605/items
  body: {"bookshelf_id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e"}


← 200
  {
    "id": "ad8e55d1-99e3-4cbf-97aa-f26c89f89605",
    "name": "My Playlist",
    "is_default": true,
    "items": [
      {
        "story_id": "36475aa6-7c7f-4bb5-8584-ed946b5b55ce",
        "source_type": "library",
        "title": "Notebook publish demo 1779089554",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
        "position": 0
      }
    ],
    "total_items": 1,
    "has_more": false
  }

  ✅ 2.3 add bookshelf story to a playlist (bookshelf_id)
→ GET http://34.44.73.189:8003/api/v1/feeds/playlists/ad8e55d1-99e3-4cbf-97aa-f26c89f89605?limit=2&offset=0


← 200
  {
    "id": "ad8e55d1-99e3-4cbf-97aa-f26c89f89605",
    "name": "My Playlist",
    "is_default": true,
    "items": [
      {
        "story_id": "36475aa6-7c7f-4bb5-8584-ed946b5b55ce",
        "source_type": "library",
        "title": "Notebook publish demo 1779089554",
        "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
        "position": 0
      }
    ],
    "total_items": 1,
    "has_more": false
  }

  ✅ 2.2 item pagination + items[].source_type populated  (1 items)


  (restored playlist: item present originally = False)


## Section 3 — Bookshelf

In [10]:
if EID:
    st, d = show("GET", f"/api/v1/feeds/bookshelf/{EID}")
    record("3.1 GET /bookshelf/{id} detail", st==200)
    record("3.3 detail includes task_id (for publishing)", "task_id" in d, f"task_id={d.get('task_id')}")
    st, m = show("GET", f"/api/v1/feeds/bookshelf/{EID}/manifest")
    record("3.1 GET /bookshelf/{id}/manifest", st==200 and isinstance(m.get("pages"), list), f"{len(m.get('pages',[]))} pages")
    st, p = show("GET", f"/api/v1/feeds/bookshelf/{EID}/pages?start_page=1&end_page=1")
    pgs = p.get("pages", [])
    record("3.1 GET /pages returns EXACTLY page 1", st==200 and len(pgs)==1 and pgs[0].get("page_index")==1, f"{len(pgs)} page(s)")
else:
    skip("3.1/3.3 bookshelf detail/manifest/pages", "no bookshelf data")

→ GET http://34.44.73.189:8003/api/v1/feeds/bookshelf/7a9df91d-8c17-4953-8e6e-a0de1081bb6e


← 200
  {
    "id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e",
    "title": "Notebook publish demo 1779089554",
    "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
    "summary": "Live demo of the user-publish path: completed task → public library.",
    "source_type": "created",
    "task_id": "49df991d-ee32-4caa-b785-1005abc014a8",
    "published_story_id": "36475aa6-7c7f-4bb5-8584-ed946b5b55ce",
    "is_published": true,
    "is_downloaded": false,
    "is_favorite": false,
    "is_liked": false,
    "total_pages": 5,
    "created_at": "2026-05-18T07:32:34.895933Z"
  }

  ✅ 3.1 GET /bookshelf/{id} detail
  ✅ 3.3 detail includes task_id (for publishing)  (task_id=49df991d-ee32-4caa-b785-1005abc014a8)
→ GET http://34.44.73.189:8003/api/v1/feeds/bookshelf/7a9df91d-8c17-4953-8e6e-a0de1081bb6e/manifest


← 200
  {
    "story_id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e",
    "title": "Notebook publish demo 1779089554",
    "total_pages": 5,
    "pages": [
      {
        "page_index": 1,
        "text": "Princess Lily loved exploring the royal garden. One sunny morning, she spotted a small, fluffy s…",
        "image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
        "audio_url": "https://storage.googleapis.com/cocobun-gpu-service-tts-output/jobs/6c0b4ac1-0de8-47f2-8a5e-c52dd…",
        "audio_duration_ms": 20180
      },
      {
        "page_index": 2,
        "text": "Together, they began their adventure. First, they found a patch of bright red strawberries. \"Let…",
        "image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
        "audio_url": "https://storage.googleapis.com/cocobun-gpu-service-tts-output/jobs/6c0b4ac1-0de8-47f2-8a5e-c52dd…",
        "audio_

← 200
  {
    "story_id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e",
    "pages": [
      {
        "page_index": 1,
        "text": "Princess Lily loved exploring the royal garden. One sunny morning, she spotted a small, fluffy s…",
        "image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
        "audio_url": "https://storage.googleapis.com/cocobun-gpu-service-tts-output/jobs/6c0b4ac1-0de8-47f2-8a5e-c52dd…",
        "audio_duration_ms": 20180
      }
    ],
    "total_pages": 5
  }

  ✅ 3.1 GET /pages returns EXACTLY page 1  (1 page(s))


In [11]:
# 3.2 favorite toggle — save & restore original state (try/finally)
if EID:
    _, det0 = _http("GET", BASE + f"/api/v1/feeds/bookshelf/{EID}")
    orig_fav = bool(det0.get("is_favorite"))
    try:
        st, d = show("POST", f"/api/v1/feeds/bookshelf/{EID}/favorite")
        record("3.2 POST favorite -> is_favorite=true", st==200 and d.get("is_favorite") is True)
        st, d = show("DELETE", f"/api/v1/feeds/bookshelf/{EID}/favorite")
        record("3.2 DELETE favorite -> is_favorite=false", st==200 and d.get("is_favorite") is False)
    finally:
        _http("POST" if orig_fav else "DELETE", BASE + f"/api/v1/feeds/bookshelf/{EID}/favorite")
        print(f"  (restored is_favorite -> {orig_fav})")
else:
    skip("3.2 favorite", "no bookshelf data")

→ POST http://34.44.73.189:8003/api/v1/feeds/bookshelf/7a9df91d-8c17-4953-8e6e-a0de1081bb6e/favorite


← 200
  {
    "id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e",
    "title": "Notebook publish demo 1779089554",
    "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
    "summary": "Live demo of the user-publish path: completed task → public library.",
    "source_type": "created",
    "task_id": "49df991d-ee32-4caa-b785-1005abc014a8",
    "published_story_id": "36475aa6-7c7f-4bb5-8584-ed946b5b55ce",
    "is_published": true,
    "is_downloaded": false,
    "is_favorite": true,
    "is_liked": false,
    "total_pages": 5,
    "created_at": "2026-05-18T07:32:34.895933Z"
  }

  ✅ 3.2 POST favorite -> is_favorite=true
→ DELETE http://34.44.73.189:8003/api/v1/feeds/bookshelf/7a9df91d-8c17-4953-8e6e-a0de1081bb6e/favorite


← 200
  {
    "id": "7a9df91d-8c17-4953-8e6e-a0de1081bb6e",
    "title": "Notebook publish demo 1779089554",
    "cover_image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/6c0b4ac1-0de8-47f2-8a5e-c…",
    "summary": "Live demo of the user-publish path: completed task → public library.",
    "source_type": "created",
    "task_id": "49df991d-ee32-4caa-b785-1005abc014a8",
    "published_story_id": "36475aa6-7c7f-4bb5-8584-ed946b5b55ce",
    "is_published": true,
    "is_downloaded": false,
    "is_favorite": false,
    "is_liked": false,
    "total_pages": 5,
    "created_at": "2026-05-18T07:32:34.895933Z"
  }

  ✅ 3.2 DELETE favorite -> is_favorite=false


  (restored is_favorite -> False)


## Section 4 — Story Manifest (request-time signing)

In [12]:
_, g = _http("GET", BASE + "/api/v1/feeds/stories?limit=1")
SID = g["items"][0]["id"] if g.get("items") else None
if SID:
    st, m1 = show("GET", f"/api/v1/feeds/stories/{SID}/manifest")
    _, m2 = _http("GET", BASE + f"/api/v1/feeds/stories/{SID}/manifest")  # 2nd call (no display)
    u1 = (m1.get("pages") or [{}])[0].get("audio_url", "")
    u2 = (m2.get("pages") or [{}])[0].get("audio_url", "")
    q1, q2 = qparams(u1), qparams(u2)
    print("  audio_url signing params:")
    for k, v in q1.items(): print(f"    {k} = {v[:28] + '…' if len(v) > 28 else v}")
    record("4.1 freshly signed + 60-min expiry", "X-Goog-Signature" in q1 and q1.get("X-Goog-Expires")=="3600", f"X-Goog-Expires={q1.get('X-Goog-Expires')}s")
    record("4.1 re-signed at request time (signature differs across two calls)",
           bool(q1.get("X-Goog-Signature")) and q1.get("X-Goog-Signature") != q2.get("X-Goog-Signature"))
else:
    skip("4.1 signed media URL", "no story")

→ GET http://34.44.73.189:8003/api/v1/feeds/stories/423d6cc3-7f42-4a9c-a1ca-caf65a45dca1/manifest


← 200
  {
    "story_id": "423d6cc3-7f42-4a9c-a1ca-caf65a45dca1",
    "title": "Princess Lily and the Royal Gardens",
    "total_pages": 5,
    "pages": [
      {
        "page_index": 1,
        "text": "Princess Lily, a graceful young deer, loved exploring the royal gardens. One sunny morning, she …",
        "image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/eb323335-bb30-4782-b9fa-d…",
        "audio_url": "https://storage.googleapis.com/cocobun-gpu-service-tts-output/jobs/eb323335-bb30-4782-b9fa-dc01e…",
        "audio_duration_ms": 14407
      },
      {
        "page_index": 2,
        "text": "She knelt gracefully, her tiny hooves careful not to disturb the delicate creatures. \"One, two, …",
        "image_url": "https://storage.googleapis.com/cocobun-gpu-service-cocobun-src/stories/eb323335-bb30-4782-b9fa-d…",
        "audio_url": "https://storage.googleapis.com/cocobun-gpu-service-tts-output/jobs/eb323335-bb30-4782-b9fa-dc01e…",
        "aud

  audio_url signing params:
    X-Goog-Algorithm = GOOG4-RSA-SHA256
    X-Goog-Credential = andy-s-dev-account@cocobun-a…
    X-Goog-Date = 20260605T061144Z
    X-Goog-Expires = 3600
    X-Goog-SignedHeaders = host
    X-Goog-Signature = 2e8fdad83e4d31eac0785a4d872f…
  ✅ 4.1 freshly signed + 60-min expiry  (X-Goog-Expires=3600s)
  ✅ 4.1 re-signed at request time (signature differs across two calls)


## Summary

In [13]:
ok = sum(1 for _, o in RESULTS if o is True); bad = sum(1 for _, o in RESULTS if o is False); sk = sum(1 for _, o in RESULTS if o is None)
print("="*60); print(f"E2E @ {BASE}:  {ok} passed, {bad} failed, {sk} skipped"); print("="*60)
for n, o in RESULTS: print(f"  {'PASS' if o is True else ('SKIP' if o is None else 'FAIL')}  {n}")
# Fail loudly (e.g. under `nbconvert --execute` / CI) on any hard failure.
assert bad == 0, f"{bad} check(s) FAILED"
if sk: print(f"\n\u26a0\ufe0f {sk} skipped (missing test data, not a failure)")

E2E @ http://34.44.73.189:8003:  16 passed, 0 failed, 0 skipped
  PASS  1.1 GET /stories/grouped returns 200 (was HTTP 500)
  PASS  1.2 search='princess' filters by title (all results contain it)
  PASS  1.3 GET /my-uploads/grouped (grouped by category)
  PASS  2.1a pagination fields (total/limit/offset/has_more)
  PASS  2.1b cover_image_url on each card
  PASS  3.2 list exposes is_favorite / is_liked / is_downloaded
  PASS  2.3 add bookshelf story to a playlist (bookshelf_id)
  PASS  2.2 item pagination + items[].source_type populated
  PASS  3.1 GET /bookshelf/{id} detail
  PASS  3.3 detail includes task_id (for publishing)
  PASS  3.1 GET /bookshelf/{id}/manifest
  PASS  3.1 GET /pages returns EXACTLY page 1
  PASS  3.2 POST favorite -> is_favorite=true
  PASS  3.2 DELETE favorite -> is_favorite=false
  PASS  4.1 freshly signed + 60-min expiry
  PASS  4.1 re-signed at request time (signature differs across two calls)
